# Notebook 02 – Train the All-Star Prediction Pipeline

**What this notebook does:**
1. Loads `data/historical_features.parquet` built by notebook 01.
2. Performs a **season-based train/test split** (no random row shuffle).
3. Builds an **imblearn Pipeline**:
   - `ColumnTransformer` → numeric imputation + scaling, categorical OHE
   - `SMOTE` (fit on train only – no leakage)
   - `XGBClassifier`
4. Evaluates with PR-AUC, ROC-AUC, Precision@24, Recall@24.
5. Saves the fitted pipeline to `models/pipeline.joblib`.

**Prerequisites**: run `notebooks/01_build_dataset.ipynb` first.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
# Seasons with year < SPLIT_YEAR are used for training;
# seasons with year >= SPLIT_YEAR are used for testing.
SPLIT_YEAR  = 2022   # train: 2008-09 … 2021-22, test: 2022-23 … 2023-24
RANDOM_STATE = 42

DATA_PATH   = REPO_ROOT / "data" / "historical_features.parquet"
MODELS_DIR  = REPO_ROOT / "models"
PIPELINE_PATH = MODELS_DIR / "pipeline.joblib"

MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} rows, {df['allstar'].sum()} All-Stars")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")
df.head(3)


## Season-based Train / Test Split

We split by season year to avoid data leakage across time.
Rows where `year < SPLIT_YEAR` go to train; the rest go to test.


In [ ]:
# Feature columns (drop metadata / target)
DROP_COLS = ["PLAYER_ID", "PLAYER_NAME", "allstar"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
NUMERIC_COLS = [c for c in FEATURE_COLS if c != "team"]
CAT_COLS = ["team"]

train_df = df[df["year"] < SPLIT_YEAR].copy()
test_df  = df[df["year"] >= SPLIT_YEAR].copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df["allstar"]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df["allstar"]

print(f"Train: {len(X_train)} rows, {y_train.sum()} All-Stars ({y_train.mean():.2%})")
print(f"Test : {len(X_test)} rows,  {y_test.sum()} All-Stars ({y_test.mean():.2%})")


## Build the Pipeline

We use `imblearn.pipeline.Pipeline` so that SMOTE is applied **only inside
the training fold** and never leaks into the test set.

```
ColumnTransformer
  ├─ numeric:      SimpleImputer(median) → StandardScaler
  └─ categorical:  OneHotEncoder(handle_unknown='ignore')
SMOTE(random_state=RANDOM_STATE)   ← train only
XGBClassifier(...)
```


In [ ]:
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

numeric_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_COLS),
        ("cat", categorical_transformer, CAT_COLS),
    ],
    remainder="drop",
)

pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

print("Pipeline constructed.")


In [ ]:
pipeline.fit(X_train, y_train)
print("Training complete.")


## Evaluation

In [ ]:
from src.evaluation import evaluate, print_evaluation

y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== Test-set evaluation ===")
results = evaluate(y_test.values, y_proba, threshold=0.5, k_values=[24, 12])
print_evaluation(results)


In [ ]:
# Per-season evaluation on test set
test_df = test_df.copy()
test_df["proba"] = y_proba

print("Per-season metrics (test seasons):")
print(f"{'Season':<12} {'P@24':>6} {'R@24':>6} {'PR-AUC':>8} {'ROC-AUC':>8} {'#AllStars':>10}")
print("-" * 58)
for yr in sorted(test_df["year"].unique()):
    mask = test_df["year"] == yr
    yt = test_df.loc[mask, "allstar"].values
    yp = test_df.loc[mask, "proba"].values
    if yt.sum() == 0:
        continue
    from src.evaluation import precision_at_k, recall_at_k
    from sklearn.metrics import average_precision_score, roc_auc_score
    p24 = precision_at_k(yt, yp, k=24)
    r24 = recall_at_k(yt, yp, k=24)
    prauc = average_precision_score(yt, yp)
    try:
        rocauc = roc_auc_score(yt, yp)
    except Exception:
        rocauc = float("nan")
    n_as = int(yt.sum())
    print(f"{yr:<12} {p24:>6.3f} {r24:>6.3f} {prauc:>8.4f} {rocauc:>8.4f} {n_as:>10}")


In [ ]:
# Top-24 predicted All-Stars in the most recent test season
latest_year = test_df["year"].max()
season_df = test_df[test_df["year"] == latest_year].copy()
season_df = season_df.sort_values("proba", ascending=False).head(24)
print(f"Top 24 predicted All-Stars for {latest_year}-{latest_year+1} season (test set):")
display_cols = ["PLAYER_NAME", "team", "proba", "allstar"]
avail_cols = [c for c in display_cols if c in season_df.columns]
print(season_df[avail_cols].to_string(index=False))


## Save Pipeline Artifact

In [ ]:
joblib.dump(pipeline, PIPELINE_PATH)
print(f"Pipeline saved to {PIPELINE_PATH}")

# Also save the feature column list for consistency checks during inference
import json
meta = {
    "feature_cols": FEATURE_COLS,
    "numeric_cols": NUMERIC_COLS,
    "cat_cols": CAT_COLS,
    "split_year": SPLIT_YEAR,
    "random_state": RANDOM_STATE,
}
with open(MODELS_DIR / "pipeline_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Metadata saved to models/pipeline_meta.json")
